In [35]:
import pandas as pd
import numpy as np

k = 8 # for n series

def clean_csv(filepath, n_series=k, temp_max=5, temp_min = -5, timestamp_threshold=1000):
    """
    Cleans a Data Streamer CSV file.
    
    Parameters:
        filepath           : path to the CSV file
        n_series           : number of series columns after the time column
        temp_max           : null temperatures above this value
        timestamp_threshold: drop timestamps > this multiple of the median gap
    
    Returns a cleaned DataFrame with columns: time_s, series_1, series_2, ...
    """
    df = pd.read_csv(filepath, skiprows=3, header=None, on_bad_lines='skip')

    time = pd.to_numeric(df.iloc[:, 0], errors='coerce') / 1000
    series = [pd.to_numeric(df.iloc[:, i+1], errors='coerce') for i in range(n_series)]

    series_names = [f'series_{i+1}' for i in range(n_series)]
    combined = pd.concat([time] + series, axis=1)
    combined.columns = ['time_s'] + series_names

    # Drop rows with missing timestamps
    combined = combined.dropna(subset=['time_s'])
    combined = combined.sort_values('time_s')

    # Remove corrupt timestamps
    time_diff = combined['time_s'].diff()
    median_gap = time_diff.median()
    combined = combined[time_diff < median_gap * timestamp_threshold]

    # Enforce monotonically increasing time
    combined = combined[combined['time_s'] > combined['time_s'].shift(1).fillna(0)]

    # Null corrupt temperature values
    for col in series_names:
        combined.loc[combined[col] > temp_max, col] = None
        combined.loc[combined[col] < temp_min, col] = None

    combined = combined.reset_index(drop=True)
    return combined


# Usage — just change n_series to however many columns you have
df = clean_csv('C:/Users/bertr/anaconda_projects/Jupyter Notebook/thermalcouple_calibration.csv', n_series=k)
df.to_csv('C:/Users/bertr/anaconda_projects/Jupyter Notebook/thermalcouple_calibration_cleaned.csv', index=False)

for col in df.columns[1:]:  # skip time_s
    print(f'{col} median: {df[col].median():.4f}')

series_1 median: -0.5600
series_2 median: -0.1900
series_3 median: -0.3700
series_4 median: -0.3100
series_5 median: -0.1200
series_6 median: 0.0000
series_7 median: 0.0000
series_8 median: 0.0000


In [5]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\bertr\anaconda_projects\Jupyter Notebook
['.ipynb_checkpoints', '2D_Particles.ipynb', '3-25_lighton_superclose-contact.csv', 'Bounce_Sim.ipynb', 'cam_obj_data.txt', 'Cleaner.ipynb', 'Cubesat_Temp_cleaning.ipynb', 'Dynamics_final_calcs.ipynb', 'Experiment_4_recording.csv', 'Friction-HW3.ipynb', 'HW06_Juan.ipynb', 'HW07_Juan.ipynb', 'HW09_Juan.ipynb', 'HW10_Juan.ipynb', 'HW2.ipynb', 'Lab 6.ipynb', 'nolighttest_cleaned(1).csv', 'nolighttest_cleaned.csv', 'no_epaper_cooled_3-13.csv', 'no_epaper_cooled_3-13_cleaned.csv', 'numberline.ipynb', 'Pendulum sim.ipynb', 'rawdata_cubesat.ipynb', 'relative_motion_data.txt', 'Relmotion_data_mani.ipynb', 'Rigid Body Equilibrium.ipynb', 'Solar_sim.ipynb', 'Temperature_graphs.ipynb', 'tracker_data_mm.txt', 'youngs_ds.png']
